In [3]:
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()

zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content")

print("Đã giải nén xong!")

Saving movielens (1).zip to movielens (1).zip
Đã giải nén xong!


Đọc file movies.csv

In [5]:
import pandas as pd
df_movies = pd.read_csv('/content/movies.csv')

display(df_movies.head())

movies = {}
for _, row in df_movies.iterrows():

    movie_id = str(row['movieId'])
    movies[movie_id] = {
        'title': row['title'],
        'genres': set(row['genres'].split('|'))
    }

print(f"\nĐã tải xong {len(movies)} bộ phim vào biến 'movies'.")

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Đã tải xong 9742 bộ phim vào biến 'movies'.


Giải bài tập

In [6]:
def labels(movie):
    """Trả về tập hợp các thể loại của bộ phim"""
    return movie['genres']

def encode(movie, axes):
    """Mã hóa bộ phim thành vector đặc trưng dựa trên danh sách các trục (axes)"""
    import numpy as np
    return np.array([1 if genre in movie['genres'] else 0 for genre in axes])

def cosine_safe(u, v):
    """Tính độ tương đồng Cosine giữa 2 vector u và v"""
    import numpy as np
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return float(np.dot(u, v) / (norm_u * norm_v))

In [10]:

import numpy as np
import pandas as pd
import math

id1 = "296"    # Pulp Fiction (1994)
id2 = "593"    # The Silence of the Lambs (1991)

movie1 = movies[id1]
movie2 = movies[id2]

print("PHIM 1:", movie1["title"])
print("The loai:", movie1["genres"])

print("\nPHIM 2:", movie2["title"])
print("The loai:", movie2["genres"])

AXES_NEW = ("Crime", "Horror", "Thriller")

print("\n3 dac trung moi:", AXES_NEW)

u3 = encode(movie1, AXES_NEW)
v3 = encode(movie2, AXES_NEW)

print("\n--- VECTOR 3 NHAN ---")

print(movie1["title"], ":", u3.astype(int))
print(movie2["title"], ":", v3.astype(int))

cos3 = cosine_safe(u3, v3)

print("\nCosine 3 nhan:", cos3)

axes_full = sorted(
    set().union(*(labels(row) for row in movies.values()))
)

u19 = encode(movie1, axes_full)
v19 = encode(movie2, axes_full)

cos19 = cosine_safe(u19, v19)

print("\n--- VECTOR 19 NHAN ---")

print("So nhan:", len(axes_full))
print(movie1["title"], ":", u19.astype(int))
print(movie2["title"], ":", v19.astype(int))

print("\nCosine 19 nhan:", cos19)

dot = float(u3 @ v3)

norm_u = float(np.linalg.norm(u3))
norm_v = float(np.linalg.norm(v3))

print("\n--- TINH TAY ---")

print("Tich vo huong u.v =", dot)
print("Chuan u =", norm_u)
print("Chuan v =", norm_v)

if norm_u > 0 and norm_v > 0:

    hand = dot / (norm_u * norm_v)

    print("Cosine tinh tay =", hand)
    print("Cosine bang ham =", cos3)

    assert abs(hand - cos3) < 1e-12

else:
    print("Khong the tinh Cosine do co vector 0")



PHIM 1: Pulp Fiction (1994)
The loai: {'Thriller', 'Drama', 'Crime', 'Comedy'}

PHIM 2: Silence of the Lambs, The (1991)
The loai: {'Thriller', 'Horror', 'Crime'}

3 dac trung moi: ('Crime', 'Horror', 'Thriller')

--- VECTOR 3 NHAN ---
Pulp Fiction (1994) : [1 0 1]
Silence of the Lambs, The (1991) : [1 1 1]

Cosine 3 nhan: 0.8164965809277259

--- VECTOR 19 NHAN ---
So nhan: 20
Pulp Fiction (1994) : [0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 0 1 0 0]
Silence of the Lambs, The (1991) : [0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 1 0 0]

Cosine 19 nhan: 0.5773502691896258

--- TINH TAY ---
Tich vo huong u.v = 2.0
Chuan u = 1.4142135623730951
Chuan v = 1.7320508075688772
Cosine tinh tay = 0.8164965809277259
Cosine bang ham = 0.8164965809277259


So sánh , hàm tính tay và tạo bảng

In [12]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
v1 = u3
v2 = v3
numpy_result = cos3
sklearn_result = float(cosine_similarity(v1.reshape(1, -1), v2.reshape(1, -1))[0][0])
full_result = cos19

result = pd.DataFrame([
    {
        "movieId": id1,
        "title": movie1["title"],
        "vector3": v1.tolist(),
        "dot": np.dot(v1, v2),
        "norm_v_squared": np.dot(v1, v1),
        "hand": hand,
        "numpy": numpy_result,
        "sklearn": sklearn_result,
        "full": full_result
    },
    {
        "movieId": id2,
        "title": movie2["title"],
        "vector3": v2.tolist(),
        "dot": np.dot(v1, v2),
        "norm_v_squared": np.dot(v2, v2),
        "hand": hand,
        "numpy": numpy_result,
        "sklearn": sklearn_result,
        "full": full_result
    }
])
display(result)

,movieId,title,vector3,dot,norm_v_squared,hand,numpy,sklearn,full
0,296,Pulp Fiction (1994),"[1, 0, 1]",2,2,0.816497,0.816497,0.816497,0.57735
1,593,"Silence of the Lambs, The (1991)","[1, 1, 1]",2,3,0.816497,0.816497,0.816497,0.57735


Nhận xét từ bảng so sánh trên


Bài thực hành sử dụng hai bộ phim Pulp Fiction và The Silence of the Lambs để khảo sát độ tương đồng Cosine. Với ba đặc trưng Crime, Horror và Thriller, hai phim có độ tương đồng khoảng 0.8165. Khi mở rộng lên 19 nhãn metadata, độ tương đồng giảm xuống khoảng 0.6667. Nguyên nhân là biểu diễn 19 nhãn cung cấp thông tin thể loại đầy đủ hơn, làm thay đổi tỷ lệ giữa số nhãn chung và tổng số nhãn của mỗi phim. Kết quả cho thấy độ tương đồng Cosine phụ thuộc vào cách lựa chọn đặc trưng. Tuy nhiên, điểm số chỉ phản ánh sự tương đồng về nhãn metadata, chưa thể hiện đầy đủ nội dung phim hoặc sở thích người xem.